# RAG Pipeline: PDF Chunking and Retrieval
This notebook builds a Retrieval-Augmented Generation pipeline over three NIST PDFs: load, chunk, embed with Azure OpenAI, store in FAISS, retrieve, and answer with GPT.

## Step 1: Load configuration
Load the Azure OpenAI settings from a shared .env file outside the project, so no API keys appear in the notebook. Then check that every required setting is present without printing secret values.

In [1]:
# Step 1: Load configuration from the shared secrets file
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.home() / ".config" / "secrets" / ".env")  # Shared secrets file used by all my projects

# Check that every setting is present, without printing the secret values
required = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_V1_URL",
    "AZURE_EMBEDDING_DEPLOYMENT",
    "AZURE_CHAT_DEPLOYMENT",
]
for name in required:
    status = "found" if os.getenv(name) else "MISSING"
    print(f"{name}: {status}")

python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 12


AZURE_OPENAI_API_KEY: found
AZURE_OPENAI_V1_URL: found
AZURE_EMBEDDING_DEPLOYMENT: found
AZURE_CHAT_DEPLOYMENT: found


### Test the Azure OpenAI connection
Send one short request to the embedding model and one to the chat model. This uses Azure's v1 endpoint, which needs no api-version, through LangChain's OpenAIEmbeddings and ChatOpenAI classes.

In [2]:
# Quick connection test using Azure's v1 API (no api-version needed)
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

azure_url = os.getenv("AZURE_OPENAI_V1_URL")
azure_key = os.getenv("AZURE_OPENAI_API_KEY")

# Stop early with a clear message if the Azure address didn't load,
# instead of silently sending requests to openai.com
if not azure_url:
    raise ValueError("AZURE_OPENAI_V1_URL is empty - check the .env file and restart the notebook")
print("Sending requests to:", azure_url)

embeddings = OpenAIEmbeddings(
    model=os.getenv("AZURE_EMBEDDING_DEPLOYMENT"),  # On Azure, "model" = your deployment name
    base_url=azure_url,                             # Send requests to Azure, not openai.com
    api_key=azure_key,                              # Use the Azure key
    check_embedding_ctx_length=False,               # Send plain text instead of pre-split tokens
)

llm = ChatOpenAI(
    model=os.getenv("AZURE_CHAT_DEPLOYMENT"),
    base_url=azure_url,
    api_key=azure_key,
    # No temperature setting: GPT-5 models only accept their default
)

test_vector = embeddings.embed_query("Hello, world")
print(f"Embedding model works: each text becomes a list of {len(test_vector)} numbers")

reply = llm.invoke("Reply with exactly: connection OK")
print(f"Chat model works: {reply.content}")


Sending requests to: https://jacobcalbillo.services.ai.azure.com/openai/v1/
Embedding model works: each text becomes a list of 1536 numbers
Chat model works: connection OK


## Step 2: Load the PDF documents
PyPDFLoader reads each PDF in the data folder and returns one Document per page. Each Document holds the page text and metadata (source file and page number).

In [6]:
# Step 2: Load every PDF in the data folder
from langchain_community.document_loaders import PyPDFLoader

DATA_DIR = Path("data")
pdf_files = sorted(DATA_DIR.glob("*.pdf"))  # Find all files ending in .pdf
print(f"Found {len(pdf_files)} PDF files\n")

all_pages = []  # An empty list that will collect pages from every PDF
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()      # One Document per page
    all_pages.extend(pages)    # Add this file's pages to the master list
    print(f"  {pdf_path.name}: {len(pages)} pages")

print(f"\nTotal pages loaded: {len(all_pages)}")

Found 3 PDF files

  NIST.AI.100-1.pdf: 48 pages
  NIST.AI.600-1.pdf: 64 pages
  NIST.SP.800-37r2.pdf: 183 pages

Total pages loaded: 295


### Inspect a sample page
Look at one loaded page to confirm the text was extracted correctly and to see its metadata.

In [18]:
# Look at one page to confirm the text came through cleanly
sample = all_pages[5]
print("Metadata:", sample.metadata)
print("\nFirst 500 characters:\n")
print(sample.page_content[:500])

Metadata: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-01-24T14:45:46-05:00', 'author': 'National Institute of Standards and Technology', 'keywords': 'Artificial Intelligence (AI); AI; AI RMF; AI RMF 1.0; AI systems; trustworthy and responsible AI.', 'moddate': '2025-06-04T13:01:45-04:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': 'As directed by the National Artificial Intelligence Initiative Act of 2020 (P.L. 116-283), the goal of the AI RMF is to offer a resource to the organizations designing, developing, deploying, or using AI systems to help manage the many risks of AI and promote trustworthy and responsible development and use of AI systems. The Framework is intended to be voluntary, rights-preserving, non-sector specific, and use-case agnostic, providing flexibility to organizations of all sizes and in all sectors and throughout society to implement the approa

## Step 3: Split the pages into chunks
RecursiveCharacterTextSplitter cuts the text into pieces of up to 1,000 characters, with 200 characters of overlap between neighbouring chunks. It splits at paragraph breaks first, then sentences, then words, so chunks stay readable.

In [8]:
# Step 3: Split pages into overlapping chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Target maximum size of each chunk, in characters
    chunk_overlap=200,  # Characters shared between neighbouring chunks
)

chunks = text_splitter.split_documents(all_pages)  # Keeps each page's metadata on its chunks
print(f"Created {len(chunks)} chunks from {len(all_pages)} pages")

Created 1230 chunks from 295 pages


## Step 4: Analyze the chunking results
Calculate the required metrics (total chunks and average chunk size), plus the smallest and largest chunk and the number of chunks per document.

In [9]:
# Step 4: Chunking metrics required by the assignment
from collections import Counter

total_chunks = len(chunks)
chunk_lengths = [len(chunk.page_content) for chunk in chunks]  # Length of each chunk, in characters
average_chunk_size = sum(chunk_lengths) / total_chunks

print(f"Total chunks:       {total_chunks}")
print(f"Average chunk size: {average_chunk_size:.0f} characters")
print(f"Smallest chunk:     {min(chunk_lengths)} characters")
print(f"Largest chunk:      {max(chunk_lengths)} characters")

# How many chunks came from each PDF
print("\nChunks per document:")
per_doc = Counter(Path(chunk.metadata["source"]).name for chunk in chunks)
for name, count in per_doc.items():
    print(f"  {name}: {count}")

Total chunks:       1230
Average chunk size: 867 characters
Smallest chunk:     76 characters
Largest chunk:      1000 characters

Chunks per document:
  NIST.AI.100-1.pdf: 150
  NIST.AI.600-1.pdf: 217
  NIST.SP.800-37r2.pdf: 863


### Display sample chunks
Print the first two chunks with their source and page to show what the pipeline will search.

In [10]:
# Show two sample chunks so the output cells display real document content
for i in [0, 1]:
    chunk = chunks[i]
    print(f"--- Chunk {i} | {Path(chunk.metadata['source']).name}, page {chunk.metadata['page_label']} ---")
    print(chunk.page_content[:400])
    print()

--- Chunk 0 | NIST.AI.100-1.pdf, page 1 ---
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)

--- Chunk 1 | NIST.AI.100-1.pdf, page 1 ---
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)
This publication is available free of charge from:
https://doi.org/10.6028/NIST.AI.100-1
January 2023
U.S. Department of Commerce
Gina M. Raimondo, Secretary
National Institute of Standards and Technology
Laurie E. Locascio, NIST Director and Under Secretary of Commerce for Standards and Technology



## Step 5: Create embeddings and store them in FAISS
Each chunk is turned into a vector of 1,536 numbers by the Azure OpenAI embedding model. FAISS stores the vectors so it can quickly find the chunks closest in meaning to a question. The index is saved to disk so it doesn't need to be rebuilt.

In [11]:
# Step 5: Embed every chunk and build the FAISS vectorstore
from langchain_community.vectorstores import FAISS

# This sends all chunks to the embedding model, so it may take a minute
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"Vectors stored in FAISS: {vectorstore.index.ntotal}")
print(f"Numbers per vector:      {vectorstore.index.d}")

# Save the index to a folder so it can be reloaded later without re-embedding
vectorstore.save_local("faiss_index")
print("Index saved to the faiss_index/ folder")

Vectors stored in FAISS: 1230
Numbers per vector:      1536
Index saved to the faiss_index/ folder


## Step 6: Semantic search
Turn a question into a vector and ask FAISS for the 3 closest chunks. The score is a distance: lower means more similar in meaning.

In [12]:
# Step 6: Find the chunks most relevant to a question
query = "What are the four functions of the AI RMF?"

results = vectorstore.similarity_search_with_score(query, k=3)  # k = how many chunks to return

print(f"Query: {query}\n")
for rank, (doc, score) in enumerate(results, start=1):
    source = Path(doc.metadata["source"]).name
    print(f"=== Result {rank} | {source}, page {doc.metadata['page_label']} | distance {score:.3f} ===")
    print(doc.page_content[:500])
    print()

Query: What are the four functions of the AI RMF?

=== Result 1 | NIST.AI.100-1.pdf, page 20 | distance 0.597 ===
and be infused throughout the other three functions.
Risk management should be continuous, timely, and performed throughout the AI system
lifecycle dimensions. AI RMF Core functions should be carried out in a way that reflects
diverse and multidisciplinary perspectives, potentially including the views of AI actors out-
side the organization. Having a diverse team contributes to more open sharing of ideas and
assumptions about purposes and functions of the technology being designed, developed,
Pa

=== Result 2 | NIST.AI.100-1.pdf, page 20 | distance 0.620 ===
NIST AI 100-1 AI RMF 1.0
Part 2: Core and Profiles
5. AI RMF Core
The AI RMF Core provides outcomes and actions that enable dialogue, understanding, and
activities to manage AI risks and responsibly develop trustworthy AI systems. As illus-
trated in Figure 5, the Core is composed of four functions: GOVERN , MAP, MEASUR

## Step 7: Retrieval-Augmented Generation
Retrieve the top chunks for a question, then pass them to GPT with instructions to answer only from that context and cite its sources. This grounds the answer in the documents and reduces made-up answers (hallucinations).

In [13]:
# Step 7: Full RAG pipeline: retrieve relevant chunks, then have GPT answer from them
def ask(question, k=3):
    # 1. Retrieve: find the k chunks closest in meaning to the question
    docs = vectorstore.similarity_search(question, k=k)

    # 2. Build the context: label each chunk with its source so GPT can cite it
    context = "\n\n".join(
        f"[Source {i}: {Path(d.metadata['source']).name}, page {d.metadata['page_label']}]\n{d.page_content}"
        for i, d in enumerate(docs, start=1)
    )

    # 3. Instructions: answer ONLY from the context, and admit when the answer isn't there
    prompt = f"""You are a document assistant. Answer the question using ONLY the context below.
If the context does not contain the answer, say "I could not find that in the documents."
Cite the sources you used, like [Source 1].

Context:
{context}

Question: {question}
Answer:"""

    # 4. Generate: send the prompt to the chat model
    response = llm.invoke(prompt)

    print(f"Question: {question}\n")
    print(response.content)
    print("\nRetrieved chunks:")
    for i, d in enumerate(docs, start=1):
        print(f"  [Source {i}] {Path(d.metadata['source']).name}, page {d.metadata['page_label']}")

### Sample queries
Three test questions, one aimed at each document, to show retrieval working across all three PDFs.

In [14]:
ask("What are the four functions of the AI RMF?")

Question: What are the four functions of the AI RMF?

The four functions are GOVERN, MAP, MEASURE, and MANAGE [Source 2].

Retrieved chunks:
  [Source 1] NIST.AI.100-1.pdf, page 20
  [Source 2] NIST.AI.100-1.pdf, page 20
  [Source 3] NIST.AI.100-1.pdf, page 42


In [15]:
ask("What is the purpose of the Prepare step in the Risk Management Framework?")

Question: What is the purpose of the Prepare step in the Risk Management Framework?

The purpose of the Prepare step is to carry out essential activities at the organization, mission and business process, and information system levels to help prepare the organization to manage its security and privacy risks using the Risk Management Framework [Source 1]. It was added to enable more effective, efficient, and cost‑effective security and privacy risk management processes [Source 3].

Retrieved chunks:
  [Source 1] NIST.SP.800-37r2.pdf, page 47
  [Source 2] NIST.SP.800-37r2.pdf, page 30
  [Source 3] NIST.SP.800-37r2.pdf, page 8


In [17]:
ask("What risks are unique to generative AI?")

Question: What risks are unique to generative AI?

The document says generative AI (GAI) “can exacerbate existing AI risks, and creates unique risks,” and that those unique risks can vary along several dimensions:
- Stage of the AI lifecycle: risks may arise during design, development, deployment, operation, and/or decommissioning.  
- Scope: risks can occur at the model/system level, the application/use‑case level, or the broader ecosystem level (examples at the ecosystem level include expansion of “algorithmic monocultures” and impacts on access to opportunity, labor markets, and the creative economies).  
- Source of risk: risks may emerge from the model’s design, training, or operation, including from model or system inputs.  
[Source 1]

Retrieved chunks:
  [Source 1] NIST.AI.600-1.pdf, page 6
  [Source 2] NIST.AI.600-1.pdf, page 5
  [Source 3] NIST.AI.600-1.pdf, page 58
